# MCTS-over-Landmarks E1 Runner

Notebook Kaggle này clone đúng nhánh `feature/mcts-landmark`, cài dependencies, rồi chạy E1a/E1b/E1c/E1d qua `scripts/kaggle_run_e1_all.py`.

Cách dùng nhanh:
- Set `HF_REPO_ID` để tải checkpoint `.pt` từ Hugging Face Dataset repo, hoặc add Kaggle Dataset chứa checkpoint.
- Chọn `PRESET = "smoke"` để test pipeline, `"report"` để chạy nghiêm túc hơn.
- Với env MuJoCo (`PointMazeMuJoCo`, `FetchPickAndPlace`, `AntMaze`), bật `INSTALL_MUJOCO = True`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Jun1801/latent_landmarks.git"
BRANCH = "feature/mcts-landmark"
REPO_DIR = Path("/kaggle/working/latent_landmarks")

# Presets: smoke | report | full
PRESET = "smoke"

# Optional: restrict tasks/experiments for faster runs.
# Examples: ONLY = ["pointmaze_numpy"] ; EXPERIMENTS = ["e1a", "e1b"]
ONLY = []
EXPERIMENTS = []

# Gate expensive E1 runs: skip a task if the clean calibrated baseline is too weak.
REQUIRE_READY = True
MIN_BASELINE_SUCCESS = 0.30
PREFLIGHT_EPISODES = 10
PREFLIGHT_CALIBRATE_EPISODES = 5

# Hugging Face Dataset repo chứa checkpoint .pt.
# Để trống nếu bạn dùng Kaggle Dataset hoặc checkpoint đã có trong repo clone.
HF_REPO_ID = "Jun1801/mcts_vla"
DOWNLOAD_CHECKPOINTS_FROM_HF = bool(HF_REPO_ID)

# Optional custom manifest. Leave as None to use the runner defaults.
MANIFEST = None

# MuJoCo install is only needed for PointMazeMuJoCo / Fetch / AntMaze.
INSTALL_MUJOCO = False

OUTPUT_ROOT = Path("/kaggle/working/e1_runs")

## Clone Repo

Cell này clone repo bằng `--single-branch --branch feature/mcts-landmark`. Nếu thư mục đã tồn tại, nó checkout nhánh đó và pull fast-forward.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

## Install Dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

if INSTALL_MUJOCO:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "mujoco>=3", "gymnasium-robotics"
    ], check=True)

print("Dependencies ready.")

## Download Checkpoints From Hugging Face

Nếu `HF_REPO_ID` được set, cell này kéo toàn bộ `*.pt` từ Hugging Face Dataset repo về `checkpoint/`.

In [ ]:
if DOWNLOAD_CHECKPOINTS_FROM_HF:
    cmd = [
        sys.executable, "scripts/hf_sync_checkpoints.py", "download",
        "--repo-id", HF_REPO_ID,
        "--checkpoint-dir", "checkpoint",
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("HF checkpoint download skipped. Set HF_REPO_ID to enable it.")

pts = sorted(Path("checkpoint").rglob("*.pt"))
print(f"Found {len(pts)} local checkpoint(s) under checkpoint/")
for p in pts[:50]:
    print(p)

## Inspect Kaggle Inputs

Runner tự tìm checkpoint theo tên trong repo và dưới `/kaggle/input`. Cell này chỉ để kiểm tra nhanh những file `.pt` đang có.

In [ ]:
input_root = Path("/kaggle/input")
if input_root.exists():
    pts = sorted(input_root.rglob("*.pt"))
    print(f"Found {len(pts)} checkpoint(s) under /kaggle/input")
    for p in pts[:50]:
        print(p)
else:
    print("/kaggle/input not found; runner will use checkpoints in the cloned repo if present.")

## Optional Custom Manifest

Nếu muốn chạy trên bộ checkpoint/dataset khác, sửa cell dưới và set `MANIFEST = str(custom_manifest)`.

In [ ]:
custom_manifest = Path("/kaggle/working/e1_tasks.json")

# Uncomment và sửa checkpoint path nếu cần.
# custom_tasks = {
#     "tasks": [
#         {
#             "name": "pointmaze_alt_seed",
#             "env": "PointMaze",
#             "checkpoint": "/kaggle/input/your-dataset/l3p_pointmaze_seed1.pt",
#             "experiments": ["e1a", "e1b", "e1c", "e1d"],
#         },
#         {
#             "name": "fetch_pick_and_place",
#             "env": "FetchPickAndPlace",
#             "checkpoint": "/kaggle/input/your-dataset/l3p_fetch.pt",
#             "experiments": ["e1a", "e1b"],
#         },
#     ]
# }
# custom_manifest.write_text(json.dumps(custom_tasks, indent=2))
# MANIFEST = str(custom_manifest)

print("MANIFEST =", MANIFEST)

## Dry Run

In command sẽ chạy, chưa chạy experiment thật.

In [ ]:
cmd = [
    sys.executable, "scripts/kaggle_run_e1_all.py",
    "--preset", PRESET,
    "--output-root", str(OUTPUT_ROOT),
    "--dry-run",
]
if MANIFEST:
    cmd += ["--manifest", MANIFEST]
if ONLY:
    cmd += ["--only", *ONLY]
if EXPERIMENTS:
    cmd += ["--experiments", *EXPERIMENTS]
if REQUIRE_READY:
    cmd += [
        "--require-ready",
        "--min-baseline-success", str(MIN_BASELINE_SUCCESS),
        "--preflight-episodes", str(PREFLIGHT_EPISODES),
        "--preflight-calibrate-episodes", str(PREFLIGHT_CALIBRATE_EPISODES),
    ]

print("$", " ".join(map(str, cmd)))
subprocess.run(cmd, check=True)

## Run E1 Batch

In [ ]:
cmd = [
    sys.executable, "scripts/kaggle_run_e1_all.py",
    "--preset", PRESET,
    "--output-root", str(OUTPUT_ROOT),
]
if MANIFEST:
    cmd += ["--manifest", MANIFEST]
if ONLY:
    cmd += ["--only", *ONLY]
if EXPERIMENTS:
    cmd += ["--experiments", *EXPERIMENTS]
if REQUIRE_READY:
    cmd += [
        "--require-ready",
        "--min-baseline-success", str(MIN_BASELINE_SUCCESS),
        "--preflight-episodes", str(PREFLIGHT_EPISODES),
        "--preflight-calibrate-episodes", str(PREFLIGHT_CALIBRATE_EPISODES),
    ]

print("$", " ".join(map(str, cmd)))
subprocess.run(cmd, check=True)

## Plot Trajectory

Cell này vẽ trajectory theo từng task trong `summary.json`: `PointMaze` dùng `plot_trajectory.py`, còn MuJoCo env (`AntMaze`, `PointMazeMuJoCo`, Fetch) dùng `plot_trajectory_mujoco.py` với đúng `--env` và checkpoint của task.

In [ ]:
summary_path = OUTPUT_ROOT / "summary.json"
mpl_dir = OUTPUT_ROOT / "mplconfig"
mpl_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_dir))

def resolve_checkpoint(name):
    p = Path(name)
    candidates = [p] if p.is_absolute() else [Path.cwd() / p]
    input_root = Path("/kaggle/input")
    if not p.is_absolute():
        candidates.extend(Path("checkpoint").rglob(p.name))
        if input_root.exists():
            candidates.extend(input_root.rglob(p.name))
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None

if not summary_path.exists():
    raise FileNotFoundError(f"Run the E1 batch first; missing {summary_path}")

summary = json.loads(summary_path.read_text())
plotted = []
for task in summary.get("tasks", []):
    name = task.get("name", "task")
    env_name = task.get("env")
    runs = task.get("runs", [])
    if runs and not any(r.get("status") == "ok" for r in runs):
        print(f"Skip trajectory for {name}: no successful experiment run.")
        continue
    ckpt = resolve_checkpoint(task.get("resolved_checkpoint") or task.get("checkpoint", ""))
    if ckpt is None:
        print(f"Skip trajectory for {name}: checkpoint not found.")
        continue

    traj_out = OUTPUT_ROOT / name / "trajectory.png"
    traj_out.parent.mkdir(parents=True, exist_ok=True)
    if env_name == "PointMaze":
        cmd = [sys.executable, "scripts/plot_trajectory.py", "--load", str(ckpt), "--out", str(traj_out)]
    else:
        cmd = [sys.executable, "scripts/plot_trajectory_mujoco.py", "--env", env_name, "--load", str(ckpt), "--out", str(traj_out)]
    print("$", " ".join(map(str, cmd)))
    try:
        subprocess.run(cmd, check=True)
        plotted.append(traj_out)
    except subprocess.CalledProcessError as e:
        print(f"Trajectory plot failed for {name} ({env_name}): exit code {e.returncode}")

print(f"Saved {len(plotted)} trajectory plot(s).")
for p in plotted:
    print(p)

## Display Plots

Hiển thị PNG thuộc các task trong `summary.json` hiện tại: E1 curves và trajectory.

In [ ]:
summary_path = OUTPUT_ROOT / "summary.json"
if summary_path.exists():
    current_tasks = {t.get("name") for t in json.loads(summary_path.read_text()).get("tasks", [])}
    pngs = sorted(
        p for p in OUTPUT_ROOT.rglob("*.png")
        if p.relative_to(OUTPUT_ROOT).parts[:1] and p.relative_to(OUTPUT_ROOT).parts[0] in current_tasks
    )
else:
    pngs = sorted(OUTPUT_ROOT.rglob("*.png"))
print(f"Found {len(pngs)} PNG plot(s).")
try:
    from IPython.display import Image, display
except ImportError:
    Image = display = None
    print("IPython is not available; listing plot paths only.")

for p in pngs:
    print(p)
    if Image is not None:
        display(Image(filename=str(p)))

## Summary

In [ ]:
summary_path = OUTPUT_ROOT / "summary.json"
print("summary:", summary_path)
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2)[:8000])
else:
    print("No summary yet. Run the E1 batch cell first.")